In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 70)

df = pd.read_parquet("./cache/dvrs_with_team_external.parquet")
print(f"Loaded {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

Loaded 92,010 rows, 56 columns


,valuation_roll_number,valuation_number_assessment,valuation_number_suffix,sale_date,district_code,sale_type,sales_group,sale_tenure,price_value_relationship,sale_price_gross,sale_price_net,sale_price_chattels,sale_price_other,capital_value,land_value,current_effective_valuation_date,situation_number,additional_situation_number,situation_name,certificate_of_title,land_area,zoning,actual_property_use,units_of_use,off_street_parking,building_age_indicator,building_condition_indicator,building_construction_indicator,building_site_coverage,building_total_floor_area,property_category,legal_description,mass_contour,mass_view,mass_scope_of_view,mass_total_floor_area,mass_deck,mass_workshop_laundry,mass_other_improvements,mass_garage_under_main_roof,mass_garage_freestanding,valuation_id_concat,longitude,latitude,in_flood_plain,in_flood_sensitive_area,dist_to_flood_plain_m,dist_to_flood_sensitive_area_m,unitary_plan_zone,linz_matched,walk_supermarket_network_m,walk_bus_stop_network_m,drive_supermarket_network_m,drive_bus_stop_network_m,noise_category_combination,sa2_new_dwellings_consented_12m
0,1,2501710000,NaN,25022026,14,S,701,1,1,1514000,1499000,15000,0,1400000,1025000,45413,166,NaN,Point Chevalier RD,335510,0,9C,91,1,1,192,GG,WI,80,87,RD192A,"Carport 1 DP 384040, Flat 1 DP 384040,",LV,N,N,87,Y,N,N,0,1,1_2501710000,174.706389,-36.865688,False,False,241.910185,4866.407069,60.0,True,1578.192284,54.442135,535.386978,0.000000,outside_current_overlay,20
1,1,2600010100,NaN,1042025,14,S,701,1,1,1515112,1507112,8000,0,1675000,1500000,44348,7,NaN,Montrose ST,NA118D/908,0.0403,9D,91,1,0,193,AA,WT,120,117,RD193B,Lot 2 DP 189123 403m2,LV,N,N,117,Y,N,N,0,0,1_2600010100,174.708779,-36.869417,False,False,194.452135,4614.989420,8.0,True,309.197383,173.432349,134.495189,155.505174,outside_current_overlay,7
2,1,4300000201,NaN,14022025,14,S,701,1,1,1622800,1609800,13000,0,1650000,1360000,44348,5,NaN,St Michaels AV,NA57C/301,0,9C,91,1,1,198,AA,WT,60,124,RD198B,"Flat 3 DP 103999, Lot 4 DP 7475 652m2",LV,N,N,124,Y,N,N,1,0,1_4300000201,174.702890,-36.861304,False,False,199.900952,4667.722058,60.0,True,1720.397633,68.370379,1122.801352,9.608944,outside_current_overlay,3
3,1,4300110000,NaN,17012025,14,S,701,1,1,2105000,2092000,13000,0,2600000,2325000,44348,35,NaN,St Michaels AV,NA405/218,0.0656,9B,91,1,1,193,AA,WT,120,123,RD193B,Lot 2 DP 15494 656m2,EF,N,N,123,Y,N,N,0,0,1_4300110000,174.700846,-36.862339,False,False,18.820703,4814.776819,18.0,True,1981.946279,329.919025,1502.064675,333.383823,outside_current_overlay,11
4,1,4300270001,NaN,11112025,14,S,701,1,2,1875000,1862000,13000,0,2300000,2025000,45413,66,NaN,St Michaels AV,115395,0.0396,9B,91,1,1,192,AA,WI,137,124,RD192B,Lot 1 DP 328302 396m2,EF,W,M,109,Y,N,N,1,0,1_4300270001,174.698847,-36.863616,False,False,6.764028,4804.594688,18.0,True,2224.730646,465.392323,1606.980701,493.788293,outside_current_overlay,9


In [ ]:
df = df.drop(columns=[
    "sale_price_net", "sale_price_chattels", "sale_price_other",
    "capital_value", "land_value",
])
print(f"Dropped sale_price_net/chattels/other, capital_value, land_value. {df.shape[1]} columns remain.")

Dropped sale_price_net/chattels/other, capital_value, land_value. 51 columns remain.


In [ ]:
def parse_sale_date(s):
    if pd.isna(s):
        return pd.NaT
    s = str(s)
    year = int(s[-4:])
    month = int(s[-6:-4])
    day = int(s[:-6])
    return pd.Timestamp(year=year, month=month, day=day)


df["sale_date_parsed"] = df["sale_date"].apply(parse_sale_date)
df["current_effective_valuation_date_parsed"] = pd.to_datetime(
    pd.to_numeric(df["current_effective_valuation_date"], errors="coerce"), unit="D", origin="1899-12-30"
)

numeric_cols = [
    "sale_price_gross",
    "land_area", "building_total_floor_area",
    "mass_total_floor_area", "building_site_coverage", "off_street_parking",
    "units_of_use", "mass_garage_under_main_roof", "mass_garage_freestanding",
    "longitude", "latitude",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Team-external numeric columns are already float/Int64, just need a plain float dtype
external_numeric_cols = [
    "walk_supermarket_network_m", "drive_supermarket_network_m",
    "walk_bus_stop_network_m", "drive_bus_stop_network_m",
    "sa2_new_dwellings_consented_12m",
]
for col in external_numeric_cols:
    df[col] = df[col].astype(float)

# Nullable boolean dtype: rows with no LINZ coordinate match (linz_matched=False)
# have None here, not True/False. Casting to "boolean" (not plain bool/object)
# makes `|` propagate that as NA instead of silently treating it as False.
for col in ["in_flood_plain", "in_flood_sensitive_area"]:
    df[col] = df[col].astype("boolean")

print("sale_date_parsed range:", df["sale_date_parsed"].min().date(), "to", df["sale_date_parsed"].max().date())
df[numeric_cols + external_numeric_cols].describe().T

sale_date_parsed range: 2024-01-01 to 2026-08-21


,count,mean,std,min,25%,50%,75%,max
sale_price_gross,92010.0,2.616173e+06,3.899396e+07,1.000000,755000.000000,1000000.000000,1.450000e+06,1.792507e+09
land_area,92010.0,2.580082e-01,4.050016e+00,0.000000,0.000000,0.028650,6.650000e-02,4.820000e+02
building_total_floor_area,92010.0,2.254900e+02,4.633720e+03,0.000000,86.000000,127.000000,1.850000e+02,3.900000e+05
mass_total_floor_area,92010.0,1.157855e+02,7.865629e+01,0.000000,77.000000,108.000000,1.520000e+02,4.888000e+03
building_site_coverage,92010.0,1.322485e+02,2.103515e+03,0.000000,59.000000,94.000000,1.370000e+02,6.000600e+05
off_street_parking,92010.0,1.570775e+00,7.167405e+00,0.000000,0.000000,1.000000,2.000000e+00,9.800000e+02
units_of_use,92010.0,1.049125e+00,7.697022e-01,0.000000,1.000000,1.000000,1.000000e+00,1.550000e+02
mass_garage_under_main_roof,92010.0,7.599826e-01,9.464774e-01,0.000000,0.000000,0.000000,2.000000e+00,5.600000e+01
mass_garage_freestanding,92010.0,3.416368e-01,6.970015e-01,0.000000,0.000000,0.000000,0.000000e+00,2.000000e+01
longitude,87916.0,1.747796e+02,1.158947e-01,174.234241,174.700457,174.765891,1.748747e+02,1.755077e+02


In [ ]:
df_cv = df.copy()
assert "capital_value" not in df_cv.columns and "land_value" not in df_cv.columns, (
    "capital_value / land_value leaked back into df_cv -- they must stay dropped "
    "from the feature set, same as the rest of this notebook"
)
print("Confirmed: capital_value / land_value are not present in df_cv")

before = len(df_cv)
df_cv = df_cv[df_cv["zoning"].astype(str).str[0].isin(["9", "1", "2"])]
print(f"Residential / rural / lifestyle zoning only: {len(df_cv):,} / {before:,} rows kept")

before = len(df_cv)
df_cv = df_cv[df_cv["units_of_use"] == 1]
print(f"Single unit of use only: {len(df_cv):,} / {before:,} rows kept")

before = len(df_cv)
df_cv = df_cv[df_cv["sale_tenure"].astype(str) == "1"]
print(f"Freehold sales only: {len(df_cv):,} / {before:,} rows kept")

before = len(df_cv)
df_cv = df_cv[df_cv["sale_type"] == "S"]
print(f"Single-property sales only: {len(df_cv):,} / {before:,} rows kept")

before = len(df_cv)
df_cv = df_cv[df_cv["price_value_relationship"] == "1"]
print(f"Arm's-length market sales only: {len(df_cv):,} / {before:,} rows kept")

before = len(df_cv)
df_cv = df_cv[df_cv["linz_matched"]]
print(f"LINZ-matched coordinate available: {len(df_cv):,} / {before:,} rows kept")

# True duplicates: same property, same sale date, same price -- these are
# the same physical sale recorded more than once (e.g. split across
# multiple legal titles/assessments under one roll number), not separate
# transactions. Rows that share a property+date but have a DIFFERENT price
# are kept -- those are genuinely separate sales (e.g. multi-unit titles).
before = len(df_cv)
df_cv = df_cv.drop_duplicates(subset=["valuation_id_concat", "sale_date_parsed", "sale_price_gross"], keep="first")
print(f"Exact duplicate sales removed (same property, date and price): {len(df_cv):,} / {before:,} rows kept")

# Sort chronologically -- required for TimeSeriesSplit below, which assumes
# row order IS time order (it does not sort for you).
df_cv = df_cv.sort_values("sale_date_parsed").reset_index(drop=True)
print(f"\nFinal filtered shape: {df_cv.shape}")
print(f"Date range: {df_cv['sale_date_parsed'].min().date()} to {df_cv['sale_date_parsed'].max().date()}")

Confirmed: capital_value / land_value are not present in df_cv
Residential / rural / lifestyle zoning only: 80,070 / 92,010 rows kept
Single unit of use only: 77,370 / 80,070 rows kept
Freehold sales only: 76,395 / 77,370 rows kept
Single-property sales only: 74,112 / 76,395 rows kept
Arm's-length market sales only: 61,244 / 74,112 rows kept
LINZ-matched coordinate available: 59,798 / 61,244 rows kept
Exact duplicate sales removed (same property, date and price): 59,711 / 59,798 rows kept

Final filtered shape: (59711, 53)


Date range: 2024-01-01 to 2026-08-18


In [ ]:
df_cv["property_type"] = df_cv["property_category"].str[0]
df_cv["sale_year"] = df_cv["sale_date_parsed"].dt.year
df_cv["sale_month"] = df_cv["sale_date_parsed"].dt.month
df_cv["sale_quarter"] = df_cv["sale_date_parsed"].dt.quarter
df_cv["has_flood_risk"] = df_cv["in_flood_plain"] | df_cv["in_flood_sensitive_area"]
df_cv["in_noise_overlay"] = df_cv["noise_category_combination"] != "outside_current_overlay"
df_cv["unitary_plan_zone"] = df_cv["unitary_plan_zone"].fillna(-1).astype(int).astype(str)

BASELINE_CAT = [
    "zoning", "building_age_indicator", "building_condition_indicator",
    "building_construction_indicator", "mass_contour", "mass_view", "mass_scope_of_view",
    "mass_deck", "mass_workshop_laundry", "mass_other_improvements",
    "property_type", "actual_property_use",
]
BASELINE_NUM = [
    "land_area", "building_total_floor_area", "mass_total_floor_area",
    "building_site_coverage", "off_street_parking",
    "mass_garage_under_main_roof", "mass_garage_freestanding",
    "sale_year", "sale_month", "sale_quarter",
]
BASELINE_BOOL = []

EXTERNAL_CAT = ["unitary_plan_zone", "noise_category_combination"]
EXTERNAL_NUM = [
    "longitude", "latitude", "dist_to_flood_plain_m", "dist_to_flood_sensitive_area_m",
    "sa2_new_dwellings_consented_12m", "walk_supermarket_network_m", "walk_bus_stop_network_m",
]
EXTERNAL_BOOL = ["in_flood_plain", "in_flood_sensitive_area", "has_flood_risk", "in_noise_overlay"]

for col in BASELINE_CAT:
    df_cv[col] = df_cv[col].fillna("Unknown").astype(str)
for col in EXTERNAL_CAT:
    df_cv[col] = df_cv[col].fillna("Unknown").astype(str)

LOG_COLS = [
    "land_area", "building_total_floor_area", "mass_total_floor_area",
    "walk_supermarket_network_m", "walk_bus_stop_network_m",
]
for col in LOG_COLS:
    df_cv[col] = np.log1p(df_cv[col])

baseline_cols = BASELINE_CAT + BASELINE_NUM + BASELINE_BOOL
augmented_cols = baseline_cols + EXTERNAL_CAT + EXTERNAL_NUM + EXTERNAL_BOOL
all_cat = BASELINE_CAT + EXTERNAL_CAT
all_num = BASELINE_NUM + EXTERNAL_NUM
all_bool = BASELINE_BOOL + EXTERNAL_BOOL

print(f"Baseline:  {len(baseline_cols)} features")
print(f"Augmented: {len(augmented_cols)} features (+{len(augmented_cols) - len(baseline_cols)} external)")

Baseline:  22 features
Augmented: 35 features (+13 external)


In [ ]:
import geopandas as gpd

# Distance to CBD: fixed reference point (Queen St / Britomart), same
# approach and same point used in cache/_scratch/build_location_factors.py.
# Computed in NZTM2000 (EPSG:2193, metres) for a real distance in metres.
points = gpd.GeoDataFrame(
    df_cv[[]], geometry=gpd.points_from_xy(df_cv["longitude"], df_cv["latitude"]), crs="EPSG:4326",
).to_crs("EPSG:2193")

CBD_LON, CBD_LAT = 174.7645, -36.8485
cbd_point = gpd.GeoSeries(gpd.points_from_xy([CBD_LON], [CBD_LAT]), crs="EPSG:4326").to_crs("EPSG:2193").iloc[0]
df_cv["distance_to_cbd"] = points.distance(cbd_point).values

print(df_cv["distance_to_cbd"].describe())

count    59711.000000
mean     17153.532314
std      11108.499645
min        730.668171
25%       9129.319716
50%      14080.002623
75%      23316.704909
max      95622.626107
Name: distance_to_cbd, dtype: float64


In [ ]:
df_cv.head()

,valuation_roll_number,valuation_number_assessment,valuation_number_suffix,sale_date,district_code,sale_type,sales_group,sale_tenure,price_value_relationship,sale_price_gross,current_effective_valuation_date,situation_number,additional_situation_number,situation_name,certificate_of_title,land_area,zoning,actual_property_use,units_of_use,off_street_parking,building_age_indicator,building_condition_indicator,building_construction_indicator,building_site_coverage,building_total_floor_area,property_category,legal_description,mass_contour,mass_view,mass_scope_of_view,mass_total_floor_area,mass_deck,mass_workshop_laundry,mass_other_improvements,mass_garage_under_main_roof,mass_garage_freestanding,valuation_id_concat,longitude,latitude,in_flood_plain,in_flood_sensitive_area,dist_to_flood_plain_m,dist_to_flood_sensitive_area_m,unitary_plan_zone,linz_matched,walk_supermarket_network_m,walk_bus_stop_network_m,drive_supermarket_network_m,drive_bus_stop_network_m,noise_category_combination,sa2_new_dwellings_consented_12m,sale_date_parsed,current_effective_valuation_date_parsed,property_type,sale_year,sale_month,sale_quarter,has_flood_risk,in_noise_overlay,distance_to_cbd
0,3770,28006,NaN,1012024,14,S,1001,1,1,1180000,44348,108,A,Morley RD,NA62D/962,0.955665,1A,21,1,0,196,AA,BT,120,4.795791,LI196B,Lot 1 DP 111743,LV,N,N,4.795791,N,N,N,0,2,3770_28006,174.769762,-37.216792,True,False,0.000000,9101.028065,16,True,8.826095,7.643731,6641.451286,2086.518284,outside_current_overlay,19.0,2024-01-01,2021-06-01,L,2024,1,1,True,False,40871.216823
1,36800,13647884,NaN,1012024,14,S,902,1,1,838000,44348,37,NaN,Paretaiuru CR,1041018,0.018331,9C,91,1,1,202,GG,XI,69,4.927254,RD202B,LOT 975 DP 572717,LV,N,N,4.795791,Y,N,N,1,0,36800_13647884,174.948115,-37.052676,False,False,40.172142,1080.899666,60,True,7.136307,5.783361,1646.997149,323.849082,outside_current_overlay,64.0,2024-01-01,2021-06-01,R,2024,1,1,False,False,27942.629620
2,35040,22701,NaN,1012024,14,S,901,1,1,1150000,44348,20,A,Redmount PL,392681,0.142281,9B,91,1,0,201,GG,WI,105,5.043425,RD201B,"Lot 2 DP 398410, Pt Lot 33 DP 89718",SF,O,S,5.043425,Y,N,N,0,0,35040_22701,174.978195,-37.073873,False,False,28.767464,381.124568,18,True,8.256333,5.301357,3666.717216,105.327014,outside_current_overlay,13.0,2024-01-01,2021-06-01,R,2024,1,1,False,False,31426.530006
3,33060,37825,NaN,1012024,14,S,606,1,1,941888,44348,141,NaN,Pooks RD,1097806,0.014002,9C,91,1,1,202,GG,FI,74,4.948760,RD202B,Lot 8 DP 583444 1/6 Sh Lot 18 DP 58344,EF,N,N,4.812184,Y,N,N,1,0,33060_37825,174.590372,-36.867883,False,False,38.416176,2705.358165,60,True,8.386835,5.542755,4305.455057,262.808203,outside_current_overlay,59.0,2024-01-01,2021-06-01,R,2024,1,1,False,False,15674.492893
4,33540,28805,NaN,1012024,14,S,608,1,1,1155632,44348,12,NaN,San Ignacio Ct,NA113C/246,0.052403,9A,91,1,2,200,GG,XT,190,5.252273,RD200B,LOT 97 DP 182345,ER,O,S,4.997212,N,N,N,2,0,33540_28805,174.611493,-36.890141,False,False,238.169911,237.062608,19,True,8.179212,6.596742,3684.865920,843.600349,outside_current_overlay,2.0,2024-01-01,2021-06-01,R,2024,1,1,False,False,14402.362770


In [ ]:
# School data (school_data.csv, NZ Schools Directory): point locations
# only -- no enrolment-zone polygon boundaries, so this is nearest-school
# distance only, not zone membership.
schools = pd.read_csv("school_data.csv", skiprows=16, header=0, dtype=str)
schools = schools[schools["Status"] == "Open"].dropna(subset=["Latitude", "Longitude"]).copy()
schools["Latitude"] = schools["Latitude"].astype(float)
schools["Longitude"] = schools["Longitude"].astype(float)
print(f"{len(schools):,} open schools with coordinates")

school_points = gpd.GeoDataFrame(
    schools[["Enrolment Scheme"]],
    geometry=gpd.points_from_xy(schools["Longitude"], schools["Latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:2193")

nearest_school = gpd.sjoin_nearest(points, school_points, distance_col="distance_to_school")
nearest_school = nearest_school[~nearest_school.index.duplicated(keep="first")]  # break exact-tie duplicates

df_cv["distance_to_school"] = nearest_school["distance_to_school"]

print(df_cv["distance_to_school"].describe())

2,564 open schools with coordinates

count    59711.000000
mean       678.943420
std        852.728418
min         17.544172
25%        330.715552
50%        505.201597
75%        725.016639
max      13460.193176
Name: distance_to_school, dtype: float64


In [ ]:
# Area-level deprivation index (NZDep2023): property -> SA2 (point-in-
# polygon against this project's SA2 2025 boundaries) -> NZDep2023 decile
# for that SA2 (1 = least deprived, 10 = most deprived). Same join as
# cache/_scratch/build_location_factors.py.
sa2_df = pd.read_csv("SA2/statistical-area-2-2025.csv", encoding="utf-8-sig", dtype=str,
                      usecols=["SA22025_V1_00", "WKT"])
sa2 = gpd.GeoDataFrame(
    sa2_df[["SA22025_V1_00"]].rename(columns={"SA22025_V1_00": "sa2_code"}),
    geometry=gpd.GeoSeries.from_wkt(sa2_df["WKT"]), crs="EPSG:4326",
).to_crs("EPSG:2193")

sa2_match = gpd.sjoin(points, sa2, how="left", predicate="within")
sa2_match = sa2_match[~sa2_match.index.duplicated(keep="first")]
df_cv["sa2_code"] = sa2_match["sa2_code"]

nzdep = pd.read_csv("./hazard_data/nzdep2023_sa2.txt", sep="\t", dtype=str)
nzdep["SA2_average_NZDep2023"] = pd.to_numeric(nzdep["SA2_average_NZDep2023"], errors="coerce")
nzdep_decile = nzdep.set_index("SA22023_code")["SA2_average_NZDep2023"]

df_cv["area_level_deprivation_index"] = df_cv["sa2_code"].map(nzdep_decile)

match_rate = df_cv["area_level_deprivation_index"].notna().sum() / df_cv["sa2_code"].notna().sum()
print(f"SA2 match rate (2025 boundary code found in NZDep2023 2023-boundary table): {match_rate:.1%}")
print(df_cv["area_level_deprivation_index"].describe())

SA2 match rate (2025 boundary code found in NZDep2023 2023-boundary table): 100.0%
count    59711.000000
mean         5.157576
std          2.966221
min          1.000000
25%          2.000000
50%          5.000000
75%          8.000000
max         10.000000
Name: area_level_deprivation_index, dtype: float64


In [ ]:
# 2023 Census SA2 totals (Stats NZ). -999 = suppressed cell -> NaN.
# Median HOUSEHOLD income is not in either file -- only median PERSONAL
# income (people aged 15+), which is what median_personal_income_by_sa is.
SA2_COL = "Statistical area 2 (SA2) 2023 code"
LF = "Subject pop: Census usually resident population count aged 15 years and over, Year: 2023, Measure: Count, Var1: Work and labour force status ({})"
INCOME = "Subject pop: Census usually resident population count aged 15 years and over, Year: 2023, Measure: Median, Var1: Total personal income (Median ($))"
POP = "Subject pop: Census usually resident population count, Year: 2023, Measure: Count, Var1: Census usually resident population count (Total)"

lf_parts = ["Employed Full-time", "Employed Part-time", "Unemployed", "Not in the Labour Force"]
census_ind = pd.read_csv(
    "2023_Census_totals_by_topic_for_individuals_by_SA2_-1302754315209467236.csv",
    usecols=[SA2_COL, INCOME] + [LF.format(p) for p in lf_parts], dtype=str,
).set_index(SA2_COL)
census_ind = census_ind.apply(pd.to_numeric, errors="coerce").replace(-999, np.nan)
census_ind = census_ind.rename(columns={LF.format(p): p for p in lf_parts} | {INCOME: "median_personal_income"})

employed = census_ind["Employed Full-time"] + census_ind["Employed Part-time"]
labour_force = employed + census_ind["Unemployed"]
working_age = labour_force + census_ind["Not in the Labour Force"]

# Stats NZ definitions: employment rate = employed / working-age pop (15+);
# unemployment rate = unemployed / labour force.
employment_rate = employed / working_age * 100
unemployment_rate = census_ind["Unemployed"] / labour_force * 100

census_pop = pd.read_csv(
    "2023_Census_totals_by_topic_for_individuals_by_SA2_3066841240893190148.csv",
    usecols=[SA2_COL, POP], dtype=str,
).set_index(SA2_COL)[POP]
census_pop = pd.to_numeric(census_pop, errors="coerce").replace(-999, np.nan)

sa2_land_area = pd.read_csv(
    "SA2/statistical-area-2-2025.csv", encoding="utf-8-sig", dtype=str,
    usecols=["SA22025_V1_00", "LAND_AREA_SQ_KM"],
).set_index("SA22025_V1_00")["LAND_AREA_SQ_KM"].astype(float)

df_cv["median_personal_income_by_sa"] = df_cv["sa2_code"].map(census_ind["median_personal_income"])
df_cv["employment_rate_by_sa"] = df_cv["sa2_code"].map(employment_rate)
df_cv["unemployment_rate_by_sa"] = df_cv["sa2_code"].map(unemployment_rate)
df_cv["population_counts_by_sa"] = df_cv["sa2_code"].map(census_pop)
df_cv["population_density"] = df_cv["population_counts_by_sa"] / df_cv["sa2_code"].map(sa2_land_area)  # people per km^2

census_cols = ["median_personal_income_by_sa", "employment_rate_by_sa", "unemployment_rate_by_sa",
               "population_counts_by_sa", "population_density"]
print(df_cv[census_cols].isna().mean().mul(100).round(2).to_frame("% missing"))
df_cv[census_cols].describe().T

                              % missing
median_personal_income_by_sa        0.0
employment_rate_by_sa               0.0
unemployment_rate_by_sa             0.0
population_counts_by_sa             0.0
population_density                  0.0


,count,mean,std,min,25%,50%,75%,max
median_personal_income_by_sa,59711.0,47041.952069,9080.960600,0.000000,40400.000000,45800.000000,53300.000000,81400.000000
employment_rate_by_sa,59711.0,66.090784,6.268225,25.900901,62.416107,66.383881,70.252572,83.333333
unemployment_rate_by_sa,59711.0,4.610638,2.342852,0.000000,3.063457,3.932584,5.383361,28.571429
population_counts_by_sa,59711.0,3027.550250,864.682102,117.000000,2511.000000,3081.000000,3609.000000,5562.000000
population_density,59711.0,2783.179953,1413.552964,2.087323,1912.744056,3018.204483,3733.041210,6917.146689


In [ ]:
import os
import shapely
from scipy.spatial import cKDTree

os.environ["OGR_GEOJSON_MAX_OBJ_SIZE"] = "0"

# Distance to coastline (LINZ NZ Coastline, land polygons), measured to the
# polygon BOUNDARY -- every onshore point is inside the landmass polygon, so
# distance-to-polygon would be 0.
# The mainland is one ~2M-vertex shape, so sjoin_nearest compared every
# property against all of it (~7 min). Instead: densify the coastline so no
# two vertices are >20 m apart, then nearest-vertex lookup with a KD-tree.
# Max error vs. the exact line distance is ~10 m (1.3 m on a 2,000-house check).
coast_polygons = gpd.read_file("./hazard_data/nz_coastline_auckland.geojson")
coast_lines = gpd.GeoSeries(coast_polygons.geometry.boundary, crs=coast_polygons.crs).to_crs("EPSG:2193")

coast_vertices = shapely.get_coordinates(shapely.segmentize(coast_lines.values, max_segment_length=20))
coast_tree = cKDTree(coast_vertices)

dist, _ = coast_tree.query(shapely.get_coordinates(points.geometry.values))
df_cv["distance_to_coastline"] = dist

print(df_cv["distance_to_coastline"].describe())

count    59711.000000
mean      1466.729641
std       1663.685464
min          3.773699
25%        411.825549
50%        958.655203
75%       1895.449362
max      16172.500924
Name: distance_to_coastline, dtype: float64


### Property's features selection:
- Floor Area
- Land Area
- Bedrooms, Bathrooms, Parking spaces,...
- Property type
- Building age, condition, construction materials,...
- Renovations,...

### Spatial and accessibility:
- Latitude and Longitude (Should be dominant here, may need to remove)
- Distance to CBD
- Distant to train station, busways,...
- Distant to shoping center, supermarket, hospital,...
- School-zone membership and distance to school

### Socioeconomic:
- Median household income by census area
- Employment and unemployment rate
- Population Density
- Area-level deprivation index

### Geographic and environment features:
- Distance to coastline, beaches, and parks
- Flood susceptibility and coastal hazard
- Other relevant hazard
- Noise exposure
- Nearby major roads


### Market and planning features
- Building concents
- Local housing suply
- Population growth
- Sale date and interest-rate environment

In [ ]:
# Features of the property itself
PROPERTY_FEATURES = [
    "land_area", "building_total_floor_area", "mass_total_floor_area",
    "off_street_parking", "mass_garage_under_main_roof", "mass_garage_freestanding",
    "property_type",
    "building_age_indicator", "building_condition_indicator", "building_construction_indicator",
]
SPATIAL_FEATURES = [
    "distance_to_cbd", "walk_supermarket_network_m", "walk_bus_stop_network_m",
    "distance_to_school", "area_level_deprivation_index",
]

SOCIOECONOMIC_FEATURES = [
    "median_personal_income_by_sa", "employment_rate_by_sa", "unemployment_rate_by_sa",
    "population_counts_by_sa", "population_density",
]

ENVIRONMENT_FEATURES = ["distance_to_coastline", "in_flood_plain", "noise_category_combination"]

# building_consents = new dwellings consented in the property's SA2, past 12 months
MARKET_FEATURES = ["sa2_new_dwellings_consented_12m", "sale_year", "sale_month", "sale_quarter"]

feature_cols = (
    ["valuation_id_concat"] + PROPERTY_FEATURES + ["latitude", "longitude"]
    + SPATIAL_FEATURES + SOCIOECONOMIC_FEATURES + ENVIRONMENT_FEATURES + MARKET_FEATURES
    + ["sale_date_parsed", "sale_price_gross"]
)
df_features = df_cv[feature_cols].rename(columns={"sa2_new_dwellings_consented_12m": "building_consents"})

# valuation_id_concat alone isn't unique -- some properties sold more than
# once in this period. Combine with the sale date to get a unique row key.
df_features["sale_id"] = (
    df_features["valuation_id_concat"] + "_" + df_features["sale_date_parsed"].dt.strftime("%Y%m%d")
)

print(f"{df_features.shape[0]:,} rows, {df_features.shape[1]} columns")
print(f"valuation_id_concat unique: {df_features['valuation_id_concat'].nunique():,} / {len(df_features):,}")
print(f"sale_id unique: {df_features['sale_id'].nunique():,} / {len(df_features):,}")
df_features.head()

59,711 rows, 33 columns
valuation_id_concat unique: 56,905 / 59,711
sale_id unique: 59,612 / 59,711


,valuation_id_concat,land_area,building_total_floor_area,mass_total_floor_area,off_street_parking,mass_garage_under_main_roof,mass_garage_freestanding,property_type,building_age_indicator,building_condition_indicator,building_construction_indicator,latitude,longitude,distance_to_cbd,walk_supermarket_network_m,walk_bus_stop_network_m,distance_to_school,area_level_deprivation_index,median_personal_income_by_sa,employment_rate_by_sa,unemployment_rate_by_sa,population_counts_by_sa,population_density,distance_to_coastline,in_flood_plain,noise_category_combination,building_consents,sale_year,sale_month,sale_quarter,sale_date_parsed,sale_price_gross,sale_id
0,3770_28006,0.955665,4.795791,4.795791,0,0,2,L,196,AA,BT,-37.216792,174.769762,40871.216823,8.826095,7.643731,2091.878779,4.0,49400.0,71.049137,3.079710,2805,52.774527,2851.276436,True,outside_current_overlay,19.0,2024,1,1,2024-01-01,1180000,3770_28006_20240101
1,36800_13647884,0.018331,4.927254,4.795791,1,1,0,R,202,GG,XI,-37.052676,174.948115,27942.629620,7.136307,5.783361,880.572763,9.0,44000.0,63.916392,8.214850,3708,3659.086539,1215.679340,False,outside_current_overlay,64.0,2024,1,1,2024-01-01,838000,36800_13647884_20240101
2,35040_22701,0.142281,5.043425,5.043425,0,0,0,R,201,GG,WI,-37.073873,174.978195,31426.530006,8.256333,5.301357,729.834901,10.0,36300.0,56.682028,11.084337,2655,2671.436413,3145.689345,False,outside_current_overlay,13.0,2024,1,1,2024-01-01,1150000,35040_22701_20240101
3,33060_37825,0.014002,4.948760,4.812184,1,1,0,R,202,GG,FI,-36.867883,174.590372,15674.492893,8.386835,5.542755,849.813692,7.0,43200.0,67.239102,5.740741,2994,4060.306482,2496.272000,False,outside_current_overlay,59.0,2024,1,1,2024-01-01,941888,33060_37825_20240101
4,33540_28805,0.052403,5.252273,4.997212,2,2,0,R,200,GG,XT,-36.890141,174.611493,14402.362770,8.179212,6.596742,956.667136,4.0,44700.0,66.326531,4.129794,3627,3378.330422,2414.200821,False,outside_current_overlay,2.0,2024,1,1,2024-01-01,1155632,33540_28805_20240101


In [ ]:
REMAKE_PATH = "./cache/remake_dataset.parquet"
df_features.to_parquet(REMAKE_PATH, index=False)

check = pd.read_parquet(REMAKE_PATH)
print(f"Saved {REMAKE_PATH}: {check.shape[0]:,} rows, {check.shape[1]} columns")

Saved ./cache/remake_dataset.parquet: 59,711 rows, 33 columns
